In [2]:
import dask.dataframe as dd
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.transforms.v2 import ToTensor, Compose, ToDtype, Resize
import warnings
import tqdm

import matplotlib.pylab as plt
warnings.filterwarnings('ignore')

In [3]:
directory="8"

# 获取所有 .pkl 文件路径
pkl_files = [f for f in os.listdir(directory) if f.endswith('.pkl')]

img_dir = os.path.join(directory, 'img')
if not os.path.exists(img_dir):
    os.makedirs(img_dir)

df_list = []
for pkl_file in tqdm.tqdm(pkl_files):
    file_path = os.path.join(directory, pkl_file)
    df = pd.read_pickle(file_path)
    df.columns = [c[1:] if c.startswith("_") else c for c in df.columns]

    # 保存 *_cam 和 wavefront 列为单独的 npz 文件，以 epoch 命名
    # 筛选包含 '_cam' 的列和 'wavefront' 列
    cam_columns = [col for col in df.columns if '_cam' in col]
    wavefront_columns = ['wavefront'] if 'wavefront' in df.columns else []

    # 遍历每一行，保存对应的 npz 文件
    for index, row in df.iterrows():
        epoch = row['epoch']
        data_dict = {}
        for col in cam_columns:
            data_dict[col] = row[col]
        for col in wavefront_columns:
            data_dict[col] = row[col]
        
        # 保存为 npz 文件
        img_path = os.path.join(img_dir, f'{epoch}.npz')
        np.savez_compressed(img_path, **data_dict)
        
    # 保存完成后drop这些列并重新保存pkl文件
    columns_to_drop = cam_columns + wavefront_columns
    df_dropped = df.drop(columns=columns_to_drop)
    df_dropped.to_pickle(file_path)
    
    df_list.append(df_dropped)
    
df_merged = pd.concat(df_list, axis=0)
df.set_index("epoch", inplace=True, drop=True)
df.to_hdf(os.path.join(directory, 'merged.h5'), key='df', mode='w')

100%|██████████| 11/11 [40:49<00:00, 222.69s/it]


ImportError: Missing optional dependency 'pytables'.  Use pip or conda to install pytables.

In [ ]:
directory="10"
"""
使用 dask 读取指定目录中的所有 .pkl 文件并合并为一个 dask DataFrame
"""
# 获取所有 .pkl 文件路径
pkl_files = [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith('.pkl')]

# 读取所有 .pkl 文件并创建 dask DataFrame 列表
ddf_list = []
for file_path in pkl_files:
    # 读取单个文件为 pandas DataFrame
    df = pd.read_pickle(file_path)
    df.columns = [c.replace('_','') for c in df.columns]
    # 转换为 dask DataFrame
    ddf = dd.from_pandas(df, npartitions=4)
    ddf_list.append(ddf)

# 合并所有 dask DataFrame
if len(ddf_list) > 1:
    combined_ddf = dd.concat(ddf_list, interleave_partitions=True)
elif len(ddf_list) == 1:
    combined_ddf = ddf_list[0]
else:
    raise ValueError("No .pkl files found in the directory")

combined_ddf = combined_ddf.set_index("epoch", drop=True)
plt.plot(combined_ddf["J"].compute())

In [11]:
# 启动 dask 客户端（可选，用于监控）
# client = Client(processes=False)  # 使用线程而非进程以减少内存开销
# print("Dask dashboard link:", client.dashboard_link)

# 计算最佳索引（J 值最小的行）
best_index = combined_ddf["J"].idxmin().compute()
worst_index = combined_ddf["J"].idxmax().compute()
# 获取最佳行的 v 值
best_v = combined_ddf.loc[best_index, "v"].compute()
worst_v = combined_ddf.loc[worst_index, "v"].compute()

In [ ]:
class DaskAOShapingDataset(Dataset):
    """
    兼容 dask 的 AOShaping 数据集类
    """
    def __init__(self, dask_data, flatten_v, transform=None):
        self.dask_data = dask_data
        self.flatten_v = flatten_v
        self.transform = transform
        # 计算数据集大小（需要使用 compute()）
        self.length = len(dask_data)
        
    def __len__(self):
        return self.length
    
    def __getitem__(self, idx):
        # 从 dask DataFrame 获取单行数据（需要使用 compute()）
        sample = self.dask_data.iloc[idx:idx+1].compute().iloc[0]
        img = sample.cam
        if img.ndim == 2:
            img = np.stack([img, img], axis=0)  # shape(2, 144, 144)
        img = img.transpose([1, 2, 0])
        v = self.flatten_v - sample.v
        v = torch.from_numpy(v[1:]).float()
        if self.transform:
            img = self.transform(img)
        return img, v

In [ ]:
# 定义数据变换
transforms = Compose([
    ToTensor(),
    ToDtype(torch.float32, scale=True),
    Resize((192, 192))
])

In [ ]:
def create_dask_data_loaders(batch_size=32, train_split=0.8):
    """
    使用 dask 创建数据加载器
    """
    # 处理数据
    data, flatten_v, best_index = process_data_with_dask()
    
    # 创建数据集
    dataset = DaskAOShapingDataset(data, flatten_v, transform=transforms)
    
    # 划分训练集和测试集
    train_size = int(train_split * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    
    # 创建数据加载器
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, test_loader, data, flatten_v, best_index

In [ ]:
# 如果直接运行此脚本，则执行数据处理
if __name__ == "__main__":
    print("开始使用 dask 处理数据...")
    data, flatten_v, best_index = process_data_with_dask()
    print(f"数据处理完成。最佳索引: {best_index}")
    print(f"数据形状: {data.shape[0].compute()} 行")
    print("前5行 J 值:")
    print(data["J"].head().compute())
    
    # 创建数据加载器
    print("创建数据加载器...")
    train_loader, test_loader, data, flatten_v, best_index = create_dask_data_loaders()
    print(f"训练集批次数量: {len(train_loader)}")
    print(f"测试集批次数量: {len(test_loader)}")